In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('tracking.csv')

print(f'Loaded {len(df)} rows')
print(df.head())

filter players/keepers

In [ ]:
# We only calculate movement for players and goalkeepers
# Referees and ball are excluded from fatigue analysis
movement_df = df[df['role'].isin(['player', 'goalkeeper'])].copy()

# Sort by player then frame — CRITICAL
# Without this, distance calculation will be wrong
movement_df = movement_df.sort_values(['player_id', 'frame']).reset_index(drop=True)

print(f'Players + goalkeepers: {len(movement_df)} rows')
print(f'Unique player IDs: {movement_df["player_id"].nunique()}')

DISTANCE MOVED PER FRAME

In [ ]:
# FPS of your video — change this if different
FPS = 25

# For each player, calculate distance moved from previous frame
# .groupby ensures we never calculate distance between two different players
movement_df['x_prev'] = movement_df.groupby('player_id')['x'].shift(1)
movement_df['y_prev'] = movement_df.groupby('player_id')['y'].shift(1)

# distance = sqrt((x2-x1)^2 + (y2-y1)^2)
movement_df['distance'] = np.sqrt(
    (movement_df['x'] - movement_df['x_prev'])**2 +
    (movement_df['y'] - movement_df['y_prev'])**2
)

# First frame of each player has no previous frame so distance = 0
movement_df['distance'] = movement_df['distance'].fillna(0)

print('Distance calculated.')
print(movement_df[['frame','player_id','x','y','distance']].head(10))

SPEED

In [ ]:
# Speed = distance / time
# time between frames = 1/FPS seconds
# So speed = distance * FPS (pixels per second)

movement_df['speed'] = movement_df['distance'] * FPS

print('Speed calculated.')
print(movement_df[['frame','player_id','distance','speed']].head(10))

print(f'\nAverage speed across all players: {movement_df["speed"].mean():.1f} px/sec')
print(f'Max speed recorded: {movement_df["speed"].max():.1f} px/sec')

acceleration

In [ ]:
# Acceleration = change in speed from previous frame
movement_df['speed_prev'] = movement_df.groupby('player_id')['speed'].shift(1)
movement_df['acceleration'] = movement_df['speed'] - movement_df['speed_prev']
movement_df['acceleration'] = movement_df['acceleration'].fillna(0)

print('Acceleration calculated.')
print(movement_df[['frame','player_id','speed','acceleration']].head(10))

speed cup

In [ ]:
# Remove unrealistic speed spikes caused by tracking errors
# A player cannot realistically move more than 200 pixels per frame
# Any speed above this is a tracking glitch, not real movement

SPEED_CAP = 200

before = len(movement_df[movement_df['speed'] > SPEED_CAP])
print(f'Rows with unrealistic speed (>{SPEED_CAP}): {before}')

# Cap the speed and recalculate acceleration
movement_df['speed'] = movement_df['speed'].clip(upper=SPEED_CAP)
movement_df['distance'] = movement_df['speed'] / FPS

# Recalculate acceleration after capping
movement_df['speed_prev'] = movement_df.groupby('player_id')['speed'].shift(1)
movement_df['acceleration'] = movement_df['speed'] - movement_df['speed_prev']
movement_df['acceleration'] = movement_df['acceleration'].fillna(0)
movement_df = movement_df.drop(columns=['speed_prev'])

print(f'Speed capped. Max speed now: {movement_df["speed"].max():.1f}')

sprint

In [ ]:
# A sprint is when a player is moving fast
# We define sprint threshold as top 25% of all speed values
# This is relative to your video so it adapts automatically

SPRINT_THRESHOLD = movement_df['speed'].quantile(0.75)
print(f'Sprint threshold: {SPRINT_THRESHOLD:.1f} px/sec')

movement_df['is_sprint'] = (movement_df['speed'] > SPRINT_THRESHOLD).astype(int)

print(f'Sprint frames: {movement_df["is_sprint"].sum()}')
print(f'Non-sprint frames: {(movement_df["is_sprint"] == 0).sum()}')

In [ ]:
# Drop the helper columns we no longer need
#movement_df = movement_df.drop(columns=['x_prev', 'y_prev', 'speed_prev'])

# Save to CSV
movement_df.to_csv('movement_features.csv', index=False)

print('=== MOVEMENT FEATURES SAVED ===')
print(movement_df[['frame','player_id','x','y','team_id','distance','speed','acceleration','is_sprint']].head(15))

In [ ]:
sample_id = movement_df['player_id'].value_counts().index[0]
p = movement_df[movement_df['player_id'] == sample_id].head(20)

print(f'=== Player #{sample_id} movement sample ===')
print(p[['frame','speed','acceleration','is_sprint']].to_string())

print(f'\n=== Per player averages ===')
summary = movement_df.groupby('player_id').agg(
    avg_speed    = ('speed', 'mean'),
    max_speed    = ('speed', 'max'),
    sprint_count = ('is_sprint', 'sum'),
    team_id      = ('team_id', 'first')
).round(2)
print(summary)